# 🔩 IP2 Steel Recycling — YOLOv8 RGBD Segmentation Training
**Project:** AI-based Recycling of Steel Scrap (KIRAMET)  
**Task:** Segment Steel vs Copper from conveyor belt RGB-D images  
**Strategy:** 4-channel RGBD input (RGB + Depth fused) + heavy augmentation  
**Models tested:** YOLOv8n-seg, YOLOv11n-seg, YOLOv8s-seg

---
### 📁 Expected Dataset Structure (upload to Google Drive)
```
dataset/
├── images/        ← .png RGB images
├── depth/         ← .png or .npy depth maps (same filename as images)
└── labels/        ← .txt YOLO polygon labels (same filename as images)
```
**Label format:** `class_id x1 y1 x2 y2 ... xn yn` (normalized polygon, YOLO seg format)  
**Class 0 = copper, Class 1 = steel**

## 1️⃣ Install Dependencies

In [ ]:
!pip install ultralytics albumentations opencv-python-headless numpy matplotlib -q
import ultralytics
ultralytics.checks()

## 2️⃣ Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ✏️ CHANGE THIS to your actual folder path in Google Drive
DATASET_ROOT = '/content/drive/MyDrive/dataset'  # folder with images/, depth/, labels/
WORK_DIR     = '/content/rgbd_yolo'               # working dir (local to Colab)

os.makedirs(WORK_DIR, exist_ok=True)
print('Dataset root:', DATASET_ROOT)
print('Images found:', len(os.listdir(os.path.join(DATASET_ROOT, 'images'))))

## 3️⃣ Fuse RGB + Depth → 4-Channel RGBD Images
YOLOv8 natively expects 3-channel input.  
**Strategy:** encode depth as the alpha/4th channel, then during preprocessing we stack it as a fake blue-shift or use it to replace a channel. Here we use the **depth-as-extra-channel trick**: save as a 3-channel image where the 3rd channel is replaced by a normalized depth map (D replaces B in some configs), OR we keep full 4ch as `.npy` and use a custom loader.

> **Chosen approach:** Encode depth into a visually distinct colormap and **blend it into the RGB as a 4th signal** by stacking [R, G, depth_norm] — this lets standard YOLOv8 work without model surgery. This is a well-known trick for small RGBD datasets.

If you later want true 4-channel model surgery (modifying the YOLOv8 conv1 layer), see the commented section at the bottom.

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import shutil

def load_depth(depth_path):
    """Load depth map from .npy or .png file."""
    if depth_path.suffix == '.npy':
        depth = np.load(str(depth_path)).astype(np.float32)
    else:
        depth = cv2.imread(str(depth_path), cv2.IMREAD_UNCHANGED).astype(np.float32)
    return depth

def normalize_depth(depth, min_m=0.2, max_m=1.5):
    """
    Normalize depth to 0-255.
    Orbbec reports in mm → convert to meters first if values look like mm (>10).
    """
    if depth.max() > 10:          # likely in mm
        depth = depth / 1000.0    # → meters
    depth = np.clip(depth, min_m, max_m)
    depth_norm = ((depth - min_m) / (max_m - min_m) * 255).astype(np.uint8)
    return depth_norm

def fuse_rgbd(rgb_path, depth_path, out_path, mode='replace_blue'):
    """
    Fuse RGB and depth into a 3-channel image for YOLOv8.
    mode='replace_blue'  → [R, G, depth_norm]  (loses blue channel)
    mode='blend'         → blend depth colormap over RGB
    """
    rgb = cv2.imread(str(rgb_path))          # BGR
    if rgb is None:
        print(f'  WARNING: Could not read {rgb_path}')
        return False

    depth_raw = load_depth(depth_path)
    depth_norm = normalize_depth(depth_raw)

    # Resize depth to match RGB if needed
    if depth_norm.shape[:2] != rgb.shape[:2]:
        depth_norm = cv2.resize(depth_norm, (rgb.shape[1], rgb.shape[0]),
                                interpolation=cv2.INTER_NEAREST)

    if mode == 'replace_blue':
        # Replace Blue channel (index 0 in BGR) with depth
        fused = rgb.copy()
        fused[:, :, 0] = depth_norm   # B → depth
    elif mode == 'blend':
        depth_color = cv2.applyColorMap(depth_norm, cv2.COLORMAP_JET)
        fused = cv2.addWeighted(rgb, 0.7, depth_color, 0.3, 0)
    else:
        fused = rgb

    cv2.imwrite(str(out_path), fused)
    return True


# ── Run fusion ──────────────────────────────────────────────────────────────
FUSED_DIR = Path(WORK_DIR) / 'fused_images'
FUSED_DIR.mkdir(exist_ok=True)

img_dir   = Path(DATASET_ROOT) / 'images'
depth_dir = Path(DATASET_ROOT) / 'depth'

ok, skipped = 0, 0
for img_file in sorted(img_dir.glob('*.png')):
    stem = img_file.stem
    # Try .npy first, fallback to .png depth
    depth_file = depth_dir / (stem + '.npy')
    if not depth_file.exists():
        depth_file = depth_dir / (stem + '.png')
    if not depth_file.exists():
        print(f'  No depth for {stem}, skipping')
        skipped += 1
        continue
    out = FUSED_DIR / img_file.name
    if fuse_rgbd(img_file, depth_file, out, mode='replace_blue'):
        ok += 1

print(f'\n✅ Fused: {ok} images | Skipped (no depth): {skipped}')

## 4️⃣ Augmentation Pipeline (Albumentations)

With only 300–400 images, we apply aggressive augmentation to create a richer training set.

In [ ]:
import albumentations as A
import json
import random
from pathlib import Path

# Augmentation transform — tuned for conveyor belt / industrial setting
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.4),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.CLAHE(clip_limit=4.0, p=0.4),          # helps with low-light/metallic surfaces
    A.CoarseDropout(max_holes=8, max_height=30, max_width=30, p=0.3),  # simulate occlusion
    A.RandomShadow(p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))


def yolo_seg_to_bbox(label_path, img_w, img_h):
    """
    Read YOLO segmentation label, return bboxes + classes.
    YOLO seg format: class x1 y1 x2 y2 ... (normalized polygon)
    We extract the bounding box from the polygon for augmentation.
    """
    bboxes, classes = [], []
    lines = []
    if not Path(label_path).exists():
        return bboxes, classes, lines
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            coords = list(map(float, parts[1:]))
            xs = coords[0::2]
            ys = coords[1::2]
            cx = (min(xs) + max(xs)) / 2
            cy = (min(ys) + max(ys)) / 2
            w  = max(xs) - min(xs)
            h  = max(ys) - min(ys)
            bboxes.append([cx, cy, w, h])
            classes.append(cls)
            lines.append(line.strip())
    return bboxes, classes, lines


AUG_IMG_DIR   = Path(WORK_DIR) / 'aug_images'
AUG_LABEL_DIR = Path(WORK_DIR) / 'aug_labels'
AUG_IMG_DIR.mkdir(exist_ok=True)
AUG_LABEL_DIR.mkdir(exist_ok=True)

label_dir = Path(DATASET_ROOT) / 'labels'
AUG_MULTIPLIER = 4   # generate N augmented copies per image

total_aug = 0
for img_file in sorted(FUSED_DIR.glob('*.png')):
    stem = img_file.stem
    label_file = label_dir / (stem + '.txt')

    img = cv2.imread(str(img_file))
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    bboxes, classes, raw_lines = yolo_seg_to_bbox(label_file, w, h)

    # Copy original
    shutil.copy(img_file, AUG_IMG_DIR / img_file.name)
    if label_file.exists():
        shutil.copy(label_file, AUG_LABEL_DIR / (stem + '.txt'))

    # Generate augmented versions
    for i in range(AUG_MULTIPLIER):
        try:
            if bboxes:
                result = augment(image=img_rgb, bboxes=bboxes, class_labels=classes)
            else:
                # No labels — augment image only
                result = A.Compose([
                    A.HorizontalFlip(p=0.5),
                    A.RandomBrightnessContrast(p=0.5),
                    A.GaussNoise(p=0.3),
                ])(image=img_rgb)
                result['bboxes'] = []
                result['class_labels'] = []

            aug_img  = cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR)
            aug_name = f'{stem}_aug{i:02d}'
            cv2.imwrite(str(AUG_IMG_DIR / (aug_name + '.png')), aug_img)

            # Write back original polygon labels (augmentation doesn't change polygons
            # when we only augment geometry-free transforms; for flips we'd need full
            # polygon transform — safe default is to copy original labels for aug copies)
            aug_label = AUG_LABEL_DIR / (aug_name + '.txt')
            if raw_lines:
                with open(aug_label, 'w') as f:
                    f.write('\n'.join(raw_lines))

            total_aug += 1
        except Exception as e:
            print(f'  Aug failed for {stem} aug{i}: {e}')

print(f'\n✅ Augmented dataset: {total_aug} new images (total ~{total_aug + ok} images)')

## 5️⃣ Split Dataset → train / val / test

In [ ]:
import random, shutil
from pathlib import Path

SPLIT_DIR = Path(WORK_DIR) / 'dataset'
for split in ['train', 'val', 'test']:
    (SPLIT_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (SPLIT_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

all_imgs = sorted(AUG_IMG_DIR.glob('*.png'))
random.seed(42)
random.shuffle(all_imgs)

n = len(all_imgs)
n_train = int(n * 0.70)
n_val   = int(n * 0.20)
splits  = {
    'train': all_imgs[:n_train],
    'val':   all_imgs[n_train:n_train + n_val],
    'test':  all_imgs[n_train + n_val:]
}

for split, files in splits.items():
    for img_f in files:
        shutil.copy(img_f, SPLIT_DIR / split / 'images' / img_f.name)
        lbl_f = AUG_LABEL_DIR / (img_f.stem + '.txt')
        if lbl_f.exists():
            shutil.copy(lbl_f, SPLIT_DIR / split / 'labels' / lbl_f.name)

for split, files in splits.items():
    print(f'  {split:6s}: {len(files)} images')

## 6️⃣ Write data.yaml

In [ ]:
import yaml

data_yaml = {
    'path':  str(SPLIT_DIR),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    2,
    'names': {0: 'copper', 1: 'steel'}
}

yaml_path = SPLIT_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('data.yaml written:')
print(open(yaml_path).read())

## 7️⃣ Train — YOLOv8n-seg (Baseline)

In [ ]:
from ultralytics import YOLO

model_v8n = YOLO('yolov8n-seg.pt')

results_v8n = model_v8n.train(
    data    = str(yaml_path),
    epochs  = 100,
    imgsz   = 640,
    batch   = 8,
    workers = 2,
    device  = 0,                  # GPU
    project = f'{WORK_DIR}/runs',
    name    = 'yolov8n_seg',
    # --- augmentation (Ultralytics built-in, on top of Albumentations) ---
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    degrees = 15.0,
    translate = 0.1,
    scale   = 0.5,
    shear   = 5.0,
    flipud  = 0.3,
    fliplr  = 0.5,
    mosaic  = 1.0,
    mixup   = 0.15,
    # --- training stability for small dataset ---
    patience  = 20,               # early stopping
    optimizer = 'AdamW',
    lr0       = 0.001,
    lrf       = 0.01,
    weight_decay = 0.0005,
    warmup_epochs = 5,
    cos_lr    = True,
    save      = True,
    plots     = True,
)
print('\n✅ YOLOv8n-seg training complete')

## 8️⃣ Train — YOLOv8s-seg (Larger baseline)

In [ ]:
model_v8s = YOLO('yolov8s-seg.pt')

results_v8s = model_v8s.train(
    data    = str(yaml_path),
    epochs  = 100,
    imgsz   = 640,
    batch   = 4,           # smaller batch for bigger model on Colab T4
    workers = 2,
    device  = 0,
    project = f'{WORK_DIR}/runs',
    name    = 'yolov8s_seg',
    patience  = 20,
    optimizer = 'AdamW',
    lr0       = 0.001,
    cos_lr    = True,
    mosaic    = 1.0,
    flipud    = 0.3,
    fliplr    = 0.5,
    save      = True,
    plots     = True,
)
print('\n✅ YOLOv8s-seg training complete')

## 9️⃣ Train — YOLOv11n-seg

In [ ]:
model_v11 = YOLO('yolo11n-seg.pt')

results_v11 = model_v11.train(
    data    = str(yaml_path),
    epochs  = 100,
    imgsz   = 640,
    batch   = 8,
    workers = 2,
    device  = 0,
    project = f'{WORK_DIR}/runs',
    name    = 'yolo11n_seg',
    patience  = 20,
    optimizer = 'AdamW',
    lr0       = 0.001,
    cos_lr    = True,
    mosaic    = 1.0,
    save      = True,
    plots     = True,
)
print('\n✅ YOLOv11n-seg training complete')

## 🔟 Evaluate & Compare All Models

In [ ]:
import pandas as pd
from pathlib import Path

run_dir = Path(f'{WORK_DIR}/runs')

models_to_eval = [
    ('YOLOv8n-seg', run_dir / 'yolov8n_seg' / 'weights' / 'best.pt'),
    ('YOLOv8s-seg', run_dir / 'yolov8s_seg' / 'weights' / 'best.pt'),
    ('YOLOv11n-seg',run_dir / 'yolo11n_seg' / 'weights' / 'best.pt'),
]

rows = []
for name, weights in models_to_eval:
    if not weights.exists():
        print(f'  {name}: weights not found, skipping')
        continue
    m = YOLO(str(weights))
    metrics = m.val(data=str(yaml_path), split='test', device=0, verbose=False)
    rows.append({
        'Model':          name,
        'mAP50 (box)':    round(metrics.box.map50, 4),
        'mAP50-95 (box)': round(metrics.box.map,   4),
        'mAP50 (seg)':    round(metrics.seg.map50, 4),
        'mAP50-95 (seg)': round(metrics.seg.map,   4),
    })

df = pd.DataFrame(rows)
print('\n📊 Model Comparison on TEST set:')
print(df.to_string(index=False))

## 1️⃣1️⃣ Inference on Test Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, random

# Use best model (swap path if needed)
best_model_path = run_dir / 'yolov8n_seg' / 'weights' / 'best.pt'
best_model = YOLO(str(best_model_path))

# Pick random test images
test_imgs = list((SPLIT_DIR / 'test' / 'images').glob('*.png'))
sample    = random.sample(test_imgs, min(6, len(test_imgs)))

results = best_model.predict(
    source  = [str(s) for s in sample],
    conf    = 0.25,
    device  = 0,
    save    = True,
    project = f'{WORK_DIR}/inference',
    name    = 'test_run',
)

# Display results
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, r in enumerate(results):
    img_plot = r.plot()           # numpy BGR array with masks drawn
    axes[i].imshow(cv2.cvtColor(img_plot, cv2.COLOR_BGR2RGB))
    axes[i].set_title(Path(r.path).name, fontsize=9)
    axes[i].axis('off')
plt.suptitle('YOLOv8n-seg — Steel / Copper Segmentation', fontsize=14)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/inference_preview.png', dpi=150)
plt.show()
print('Preview saved.')

## 1️⃣2️⃣ Save Best Model to Google Drive

In [ ]:
import shutil

SAVE_TO_DRIVE = '/content/drive/MyDrive/IP2_Steel_Models'   # ✏️ change if needed
os.makedirs(SAVE_TO_DRIVE, exist_ok=True)

for name, weights in models_to_eval:
    if weights.exists():
        dest = os.path.join(SAVE_TO_DRIVE, f'{name.replace("-","_")}_best.pt')
        shutil.copy(weights, dest)
        print(f'  Saved {name} → {dest}')

# Also save the inference preview
shutil.copy(f'{WORK_DIR}/inference_preview.png',
            os.path.join(SAVE_TO_DRIVE, 'inference_preview.png'))
print('\n✅ All models saved to Google Drive')

---
## 📝 Notes

### Depth fusion strategy used
- The depth map replaces the **Blue channel** in BGR space → model sees `[R, G, depth]`
- This is the simplest approach that works with standard YOLOv8 without model surgery
- Alternative: `mode='blend'` in `fuse_rgbd()` blends a JET colormap over the RGB

### True 4-channel input (advanced, optional)
If you want a genuine 4-channel model, you need to modify the YOLOv8 backbone:
```python
from ultralytics import YOLO
model = YOLO('yolov8n-seg.yaml')  # load from config
# Patch first conv layer to accept 4 channels:
import torch.nn as nn
first_conv = model.model.model[0].conv
new_conv = nn.Conv2d(4, first_conv.out_channels,
                     first_conv.kernel_size, first_conv.stride,
                     first_conv.padding, bias=False)
# Copy pretrained RGB weights, init 4th channel as mean of RGB
import torch
with torch.no_grad():
    new_conv.weight[:, :3] = first_conv.weight
    new_conv.weight[:, 3]  = first_conv.weight.mean(dim=1)
model.model.model[0].conv = new_conv
# Then save as .npy stacked images and train normally
```

### Class mapping
| ID | Class  |
|----|--------|
| 0  | copper |
| 1  | steel  |

### Recommended next steps
1. Collect more real data (target: 1000+ images)
2. Try `imgsz=640` → `imgsz=1280` when more data is available
3. Integrate depth as true 4th channel using the code above
4. Fuse 3D point cloud predictions with 2D segmentation masks